# Diabetes Readmission RL Project
# Approche : Reinforcement Learning Trees (RLT) – Zhu et al., 2015

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
sns.set(style="whitegrid")

## Business Understanding

In [1]:
print("Business Understanding")
print("BO1 — Valider RLT en haute dimension et sparsité")
print(" Évaluer RLT sur datasets variés pour applications industrielles.")
print("BO2 — Optimiser les modèles d'apprentissage")
print("Améliorer la précision et la stabilité des prédictions en ML .")
print("BO3 — Répliquer et étendre la recherche académique")
print("Confirmer les résultats de Zhu et al. (2015) avec implémentation moderne.")

Business Understanding
BO1 — Valider RLT en haute dimension et sparsité
 Évaluer RLT sur datasets variés pour applications industrielles.
BO2 — Optimiser les modèles d'apprentissage
Améliorer la précision et la stabilité des prédictions en ML .
BO3 — Répliquer et étendre la recherche académique
Confirmer les résultats de Zhu et al. (2015) avec implémentation moderne.


## Data Science Objectives

In [2]:
print("\nData Science Objectives")

print("DSO1 —Prédire la cible sur chaque dataset")
print("Régression/Classification avec RLT vs baselines")

print("DSO2 —Comparer les performances")
print("RMSE/Accuracy moyen sur 50 runs.")

print("DSO3 —Analyser l'impact du Variable Muting ")
print("Test avec/sans muting pour confirmer la robustesse.")


Data Science Objectives
DSO1 —Prédire la cible sur chaque dataset
Régression/Classification avec RLT vs baselines
DSO2 —Comparer les performances
RMSE/Accuracy moyen sur 50 runs.
DSO3 —Analyser l'impact du Variable Muting 
Test avec/sans muting pour confirmer la robustesse.


## Data Understanding

 Data Understanding
Les 10 datasets UCI représentent des domaines variés (économie, médecine, environnement). 

| Dataset | Type | Instances | Features | Cible |
|---------|------|-----------|----------|-------|
| Boston Housing | Régression | 506 | 13 | Prix logement |
| Parkinson | Régression | 5 875 | 20 | Score UPDRS |
| Sonar | Classification | 208 | 60 | Mine/Roche |
| White Wine | Régression | 4 898 | 11 | Qualité |
| Red Wine | Régression | 1 599 | 11 | Qualité |
| Parkinson Oxford | Classification | 195 | 22 | Maladie/Sain |
| Ozone | Classification | 2 536 | 73 | Niveau ozone |
| Concrete | Régression | 1 030 | 9 | Résistance |
| Breast Cancer | Classification | 569 | 30 | Malin/Bénin |
| Auto MPG | Régression | 398 | 8 | Consommation |



## Data Preparation

In [ ]:
from sklearn.datasets import load_boston, load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Pipeline de pré-traitement (universel pour les 10 datasets)
def pipeline_preprocess(df, target_col=None, task="regression"):
    if target_col:
        y = df[target_col].copy()
        X = df.drop(columns=[target_col])
    else:
        X = df.copy()
        y = None
    
    # 1. Remplacer ? par NaN
    X = X.replace("?", np.nan)
    
    # 2. LabelEncoder pour catégorielles
    X = pd.DataFrame(X)
    for col in X.select_dtypes(include='object').columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # 3. Imputation médiane
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    
    # 4. StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    if task == "classification" and y is not None:
        y = LabelEncoder().fit_transform(y)
    
    return X, y, scaler

# Classe RLT simplifiée (fidèle à l'article)
class SimpleRLT:
    def __init__(self, max_depth=6, mute_ratio=0.5):
        self.max_depth = max_depth
        self.mute_ratio = mute_ratio

    def fit(self, X, y):
        self.tree = self._build(X, y, 0, list(range(X.shape[1])))

    def _gini(self, y):
        p = np.bincount(y.astype(int)) / len(y)
        return 1 - np.sum(p**2)

    def _best_split(self, X, y, features):
        best_gain = -1
        best_f, best_v = None, None
        for f in np.random.choice(features, min(10, len(features)), replace=False):
            vals = np.unique(X[:, f])[:5]
            for v in vals:
                mask = X[:, f] <= v
                if len(y[mask]) < 5 or len(y[~mask]) < 5: continue
                gain = self._gini(y) - (len(y[mask])/len(y))*self._gini(y[mask]) - (len(y[~mask])/len(y))*self._gini(y[~mask])
                if gain > best_gain:
                    best_gain, best_f, best_v = gain, f, v
        return best_f, best_v

    def _build(self, X, y, depth, features):
        if depth >= self.max_depth or len(np.unique(y)) == 1:
            return np.bincount(y.astype(int)).argmax()

        f, v = self._best_split(X, y, features)
        if f is None:
            return np.bincount(y.astype(int)).argmax()

        mask = X[:, f] <= v
        muted = features if len(features) < 5 else np.random.choice(features, int(len(features)*(1-self.mute_ratio)), replace=False)
        left = self._build(X[mask], y[mask], depth+1, muted)
        right = self._build(X[~mask], y[~mask], depth+1, muted)
        return (f, v, left, right)

    def predict(self, X):
        def _pred(x, node):
            if not isinstance(node, tuple): return node
            f, v, left, right = node
            return _pred(x, left) if x[f] <= v else _pred(x, right)
        return [_pred(x, self.tree) for x in X]

# Benchmark (exemple sur 3 datasets pour test - étendre aux 10)
results = []
for name, task, data in [("Boston", "regression", load_boston(return_X_y=True)), ("Breast Cancer", "classification", load_breast_cancer(return_X_y=True)), ("Wine", "regression", fetch_openml("wine-quality", version=1, as_frame=False)) ]:
    X, y = data
    X, y = pipeline_preprocess(X, y, task)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3)
    
    # RLT
    preds = []
    for _ in range(10):
        idx = np.random.choice(len(X_tr), len(X_tr), replace=True)
        tree = SimpleRLT()
        tree.fit(X_tr[idx], y_tr[idx])
        preds.append(tree.predict(X_te))
    y_pred_rlt = np.mean(preds, axis=0) if task == "regression" else np.apply_along_axis(lambda x: np.bincount(x.astype(int)).argmax(), 0, np.array(preds))
    score_rlt = mean_squared_error(y_te, y_pred_rlt, squared=False) if task == "regression" else accuracy_score(y_te, y_pred_rlt)
    
    # RF baseline
    rf = RandomForestRegressor(n_estimators=50) if task == "regression" else RandomForestClassifier(n_estimators=50)
    rf.fit(X_tr, y_tr)
    y_pred_rf = rf.predict(X_te)
    score_rf = mean_squared_error(y_te, y_pred_rf, squared=False) if task == "regression" else accuracy_score(y_te, y_pred_rf)
    
    results.append((name, score_rlt, score_rf))
    print(f"{name} → RLT: {score_rlt:.4f} | RF: {score_rf:.4f}")

print("Benchmark terminé ! Étendre aux 10 datasets.")